In [ ]:
# Import custom helper function that computes trip duration (in minutes)
# from started_at/ended_at — encapsulated so the logic can be reused and unit tested
from citibike.citibike_utils import get_trip_duration_mins

# Import helper to convert a timestamp column into just a date column (used for trip_start_date)
from utils.datetime_utils import timestamp_to_date_col

# Functions to build the metadata map column (same pattern as in the Bronze notebook)
from pyspark.sql.functions import create_map, lit

In [ ]:
# Read the job parameters passed in via base_parameters in the bundle's job.yml.
# These widgets only resolve to real values when the notebook runs as part of an
# actual job (e.g. via "databricks bundle run") — running the cell standalone
# in the notebook editor won't have these widgets set.
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")

# Dynamically resolves to citibike_dev/test/prod,
# depending on which target this job was deployed to
catalog = dbutils.widgets.get("catalog")  

In [ ]:
# Enable autoreload so changes made to local modules (e.g. src/citibike/citibike_utils.py)
# are automatically picked up without needing to restart the kernel every time
%load_ext autoreload
%autoreload 2

In [ ]:
# Read from the Bronze table using the dynamic catalog variable,
# so this notebook works correctly regardless of which environment (dev/test/prod) it runs in
df = spark.read.table(f"{catalog}.01_bronze.jc_citibike")

In [ ]:
# Compute trip duration in minutes and add it as a new column,
# using the reusable helper function imported from src/citibike/citibike_utils.py
df = get_trip_duration_mins(df, "started_at", "ended_at", "trip_duration_mins")

In [ ]:
# Derive trip_start_date (date only, no time) from started_at —
# needed later for daily aggregations in the Gold layer (daily_ride_summary)
df = timestamp_to_date_col(df, "started_at", "trip_start_date")

In [ ]:
# Add metadata map column for pipeline lineage/observability — same pattern as in Bronze notebook.
# Values are placeholders for now, to be replaced with real job/run identifiers
# once this runs as part of an actual Databricks job
df = df.withColumn("metadata", create_map(
    lit("pipeline_id"), lit(pipeline_id),
    lit("run_id"), lit(run_id),
    lit("task_id"), lit(task_id),
    lit("processed_timestamp"), lit(processed_timestamp)
))

In [ ]:
# Keep only the columns relevant for the Silver layer (drop raw lat/lng and station IDs
# that were needed in Bronze but aren't part of the Silver business-level schema)
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
)

In [ ]:
# Write the Silver-layer DataFrame as a managed Delta table,
# same overwrite pattern as Bronze — still iterating on structure during development.
# Uses the dynamic "catalog" variable
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.02_silver.jc_citibike")